# RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain implementation of a RAG pipeline.

Using the VectorStore created previously with Hugging Face `all-MiniLM-L6-v2` embeddings, and using Gemini `gemini-2.5-flash-lite` for generation.

### Install the Gemini integration

**Note:** This package provides LangChain's integration with Google's Gemini API. Run this in your terminal inside your project virtual environment.

In [3]:
pip install -U langchain-google-genai gradio

  Using cached pillow-12.3.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 348.2 kB/s  0:02:41m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 238.5 kB/s  0:00:40eta 0:00:02
Using cached pillow-12.3.0-cp314-cp314-macosx_11_0_arm64.whl (4.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.1/863.1 kB 359.6 kB/s  0:00:02eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [gradio]15/16 [gradio]]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Import the required libraries

**Note:** `ChatGoogleGenerativeAI` connects LangChain to Gemini. Chroma is the vector store, while `HuggingFaceEmbeddings` is kept from Day 2 because the same embedding model must be used for both stored documents and incoming queries.

In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

### Configure the model and database

**Note:** `MODEL` is the Gemini model used for generation, while `DB_NAME` points to the Chroma vector store created in the previous notebook. `load_dotenv()` loads `GOOGLE_API_KEY` from your `.env` file.

In [8]:
MODEL = "gemini-3.1-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

**Note:** The embedding model remains exactly the same as Day 2. The stored document vectors were created with `all-MiniLM-L6-v2`, so the query must also be embedded using the same model and vector space.

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13930.07it/s]


### Set up the 2 key LangChain objects: retriever and LLM

**Note:** The `retriever` searches Chroma for relevant chunks. The `llm` is Gemini, which takes the retrieved context and generates the final answer. `temperature=0` makes generation more deterministic by favoring the highest-probability token choices.

In [10]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

**Note:** `invoke()` is the common LangChain interface for running these components. The retriever returns relevant `Document` objects, while the LLM returns Gemini's generated response.

In [11]:
retriever.invoke("Who is Avery?")

[Document(id='255073e2-ec93-4987-a716-1d6846d9189c', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

**Note:** This tests Gemini independently of RAG. At this point Gemini is not receiving anything from Chroma; it is simply answering the question using its own model knowledge.

In [12]:
llm.invoke("Who is Avery?")

AIMessage(content=[{'type': 'text', 'text': 'Because "Avery" is a common name, it could refer to several different people or entities depending on the context. Here are the most likely possibilities:\n\n### 1. Fictional Characters\n*   **Avery Bullock:** A recurring character in the animated series *American Dad!*. He is the eccentric and often unhinged Deputy Director of the CIA.\n*   **Avery Jessup:** A character from the TV show *30 Rock*, played by Elizabeth Banks. She is a high-powered news anchor and the wife of Jack Donaghy.\n*   **Avery Barkley:** A character from the musical drama series *Nashville*, played by Jonathan Jackson.\n*   **Avery Samuels:** A character from the horror anthology series *Scream Queens*.\n\n### 2. Historical Figures\n*   **Oswald Avery:** A famous Canadian-American physician and medical researcher. He is best known for the "Avery–MacLeod–McCarty experiment" (1944), which proved that DNA is the substance that causes genetic transformation, a foundationa

In [13]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

### Build the complete RAG function

**Note:** The function first retrieves relevant chunks, combines them into context, places that context into the system prompt, and finally sends the context plus the user's question to Gemini.

In [14]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

### Test the complete RAG pipeline

**Note:** Unlike the earlier direct Gemini call, this question now goes through retrieval first. The relevant Chroma chunks become context for Gemini.

In [15]:
answer_question("Who is Averi Lancaster?", [])

[{'type': 'text',
  'text': "Hello! I'd be happy to tell you about Avery Lancaster.\n\nAvery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. Based in San Francisco, she has been leading the company since she co-founded it in 2015. Under her guidance, Insurellm has grown into a leading Insurance Tech provider. Avery is widely recognized for her innovative leadership strategies and her deep expertise in risk management, which have been instrumental in bringing the company into the mainstream insurance market.\n\nBefore founding Insurellm, Avery served as a Senior Product Manager at Innovate Insurance Solutions from 2013 to 2015, where she focused on developing insurance products for the tech sector.\n\nIs there anything else you'd like to know about our leadership team?",
  'extras': {'signature': 'EnEKbwERTTIPuhW/gQ2HzvjPp/9+adztd2Q3iCJLADnLUO5hSAeAZiiL9agg8Os+k8si46s+XIp6gXkpaUe0QP9K4l9n3T/hs+0lNQW98hHRnspruHd8BnLLH/MfFD6IVqarUqA1kc/yBF8NnBfD5gNuRg=='}}]


The complete pipeline is now:

```text
User Question
      ↓
all-MiniLM-L6-v2
      ↓
Query Embedding
      ↓
Chroma Similarity Search
      ↓
Relevant Chunks
      ↓
Context + Question
      ↓
Gemini 2.5 Flash-Lite
      ↓
Final Answer
```

### Add a simple chat interface

**Note:** Gradio provides a chat UI around the same `answer_question()` function. The RAG pipeline itself does not change.

In [16]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
